<a href="https://colab.research.google.com/github/Rollybuilds/Quiz-management-System/blob/main/Quiz_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PROJECT 6 - QUIZ MANAGEMENT SYSTEM**

In [1]:
import csv
import os
from abc import ABC, abstractmethod
from datetime import datetime

QUESTIONS_FILE = "questions.csv"
RESULTS_FILE = "results.csv"
QUESTION_FIELDS = ["type", "question", "option_a", "option_b",
                   "option_c", "option_d", "answer"]

# Sample questions, used to create questions.csv if the file does not exist
SAMPLE_QUESTIONS = [
    ["MCQ", "Which keyword is used to define a function in Python?",
     "function", "def", "fun", "define", "B"],
    ["MCQ", "Which of these data types is mutable?",
     "tuple", "string", "list", "int", "C"],
    ["TF", "Python is a case-sensitive language.", "", "", "", "", "T"],
    ["MCQ", 'What is the output of len("Python")?',
     "5", "6", "7", "Error", "B"],
    ["TF", "A tuple can be changed after it is created.", "", "", "", "", "F"],
    ["MCQ", "Which module is used to create abstract classes?",
     "abc", "os", "csv", "math", "A"],
]


# ---------------------------------------------------------------- Exception
class QuizError(Exception):
    """Raised when a quiz action cannot be completed."""


# ------------------------------------------------- Abstract base class (abc)
class Question(ABC):
    """Base class for all types of questions."""

    def __init__(self, text, correct_answer):
        self.__text = ""                  # private attributes (encapsulation)
        self.__correct_answer = ""
        self.text = text                  # uses the setters below for validation
        self.correct_answer = correct_answer

    # Getters and setters
    @property
    def text(self):
        return self.__text

    @text.setter
    def text(self, value):
        if not value.strip():
            raise ValueError("Question text cannot be empty.")
        self.__text = value.strip()

    @property
    def correct_answer(self):
        return self.__correct_answer

    @correct_answer.setter
    def correct_answer(self, value):
        value = value.strip().upper()
        if value not in self.valid_answers():
            raise ValueError("Correct answer is not valid for this question.")
        self.__correct_answer = value

    # Common actions
    def check_answer(self, answer):
        return answer.strip().upper() == self.correct_answer

    def ask(self):
        """Show the question, read a valid answer, and return True if correct."""
        self.display()
        while True:
            answer = input("Your answer: ").strip().upper()
            if answer in self.valid_answers():
                return self.check_answer(answer)
            print(f"Please enter one of: {', '.join(self.valid_answers())}")

    # Every child class must implement these
    @abstractmethod
    def valid_answers(self):
        """Return the list of answers the user is allowed to enter."""

    @abstractmethod
    def display(self):
        """Print the question on the screen."""


# ------------------------------------------------ Derived classes (inheritance)
class MultipleChoiceQuestion(Question):
    def __init__(self, text, options, correct_answer):
        super().__init__(text, correct_answer)
        if len(options) != 4 or not all(option.strip() for option in options):
            raise ValueError("A multiple choice question needs 4 options.")
        self.__options = options

    def valid_answers(self):
        return ["A", "B", "C", "D"]

    def display(self):
        print(self.text)
        for letter, option in zip("ABCD", self.__options):
            print(f"  {letter}. {option}")


class TrueFalseQuestion(Question):
    def valid_answers(self):
        return ["T", "F"]

    def display(self):
        print(self.text)
        print("  T = True, F = False")


# ---------------------------------------------------------------- Quiz
class Quiz:
    """Loads questions, conducts the quiz, and saves the results."""

    def __init__(self):
        self.__questions = []

    @property
    def question_count(self):
        return len(self.__questions)

    # ----- file handling
    def __create_sample_file(self):
        with open(QUESTIONS_FILE, "w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(QUESTION_FIELDS)
            writer.writerows(SAMPLE_QUESTIONS)

    def __build_question(self, row):
        if row["type"] == "MCQ":
            options = [row["option_a"], row["option_b"],
                       row["option_c"], row["option_d"]]
            return MultipleChoiceQuestion(row["question"], options, row["answer"])
        if row["type"] == "TF":
            return TrueFalseQuestion(row["question"], row["answer"])
        raise ValueError("Unknown question type.")

    def load_questions(self):
        """Read the questions from the CSV file. Returns the number loaded."""
        try:
            if not os.path.exists(QUESTIONS_FILE):
                self.__create_sample_file()
                print(f"{QUESTIONS_FILE} not found. A sample file was created.")

            questions, skipped = [], 0
            with open(QUESTIONS_FILE, newline="") as file:
                for row in csv.DictReader(file):
                    try:
                        questions.append(self.__build_question(row))
                    except (KeyError, ValueError):
                        skipped += 1
        except OSError:
            raise QuizError("Could not read the questions file.")

        if skipped:
            print(f"{skipped} invalid question(s) were skipped.")
        if not questions:
            raise QuizError("No valid questions found in the file.")
        self.__questions = questions
        return len(questions)

    def save_result(self, name, score):
        total = self.question_count
        percentage = score / total * 100
        is_new_file = not os.path.exists(RESULTS_FILE)
        try:
            with open(RESULTS_FILE, "a", newline="") as file:
                writer = csv.writer(file)
                if is_new_file:
                    writer.writerow(["name", "date", "score", "total", "percentage"])
                writer.writerow([name, datetime.now().strftime("%d-%m-%Y %H:%M"),
                                 score, total, f"{percentage:.1f}"])
        except OSError:
            print("Error: could not save the result to file.")

    def load_results(self):
        try:
            with open(RESULTS_FILE, newline="") as file:
                return list(csv.DictReader(file))
        except FileNotFoundError:
            return []

    # ----- quiz actions
    def conduct_quiz(self):
        """Ask every question and return the score."""
        if not self.__questions:
            raise QuizError("Please load the questions first (option 1).")

        score = 0
        for number, question in enumerate(self.__questions, start=1):
            print(f"\nQuestion {number} of {self.question_count}")
            if question.ask():
                print("Correct!")
                score += 1
            else:
                print(f"Wrong! The correct answer is {question.correct_answer}.")
        return score


# ---------------------------------------------------------------- Menu
def feedback(percentage):
    if percentage >= 80:
        return "Excellent!"
    if percentage >= 50:
        return "Good job!"
    return "Keep practicing!"


def main():
    quiz = Quiz()

    menu = """
===== QUIZ MANAGEMENT SYSTEM =====
1. Load questions from file
2. Start quiz
3. View saved results
4. Exit
"""

    while True:
        print(menu)
        choice = input("Enter your choice (1-4): ").strip()

        try:
            if choice == "1":
                count = quiz.load_questions()
                print(f"{count} questions loaded successfully.")

            elif choice == "2":
                if quiz.question_count == 0:
                    raise QuizError("Please load the questions first (option 1).")
                name = input("Enter your name: ").strip()
                if not name:
                    raise QuizError("Name cannot be empty.")
                score = quiz.conduct_quiz()
                percentage = score / quiz.question_count * 100
                print("\n----- RESULT -----")
                print(f"Name   : {name}")
                print(f"Score  : {score} / {quiz.question_count}")
                print(f"Percent: {percentage:.1f}%")
                print(feedback(percentage))
                quiz.save_result(name, score)
                print(f"Result saved in {RESULTS_FILE}.")

            elif choice == "3":
                results = quiz.load_results()
                if not results:
                    print("No results saved yet.")
                else:
                    print(f"{'Name':<15}{'Date':<18}{'Score':>8}{'Percent':>10}")
                    for r in results:
                        score_text = f"{r['score']}/{r['total']}"
                        print(f"{r['name']:<15}{r['date']:<18}"
                              f"{score_text:>8}{r['percentage'] + '%':>10}")

            elif choice == "4":
                print("Goodbye!")
                break

            else:
                print("Invalid choice. Please select 1-4.")

        except QuizError as error:
            print(f"Error: {error}")


if __name__ == "__main__":
    main()


===== QUIZ MANAGEMENT SYSTEM =====
1. Load questions from file
2. Start quiz
3. View saved results
4. Exit

Enter your choice (1-4): 1
questions.csv not found. A sample file was created.
6 questions loaded successfully.

===== QUIZ MANAGEMENT SYSTEM =====
1. Load questions from file
2. Start quiz
3. View saved results
4. Exit

Enter your choice (1-4): 2
Enter your name: Rolly

Question 1 of 6
Which keyword is used to define a function in Python?
  A. function
  B. def
  C. fun
  D. define
Your answer: b
Correct!

Question 2 of 6
Which of these data types is mutable?
  A. tuple
  B. string
  C. list
  D. int
Your answer: c
Correct!

Question 3 of 6
Python is a case-sensitive language.
  T = True, F = False
Your answer: True
Please enter one of: T, F
Your answer: T
Correct!

Question 4 of 6
What is the output of len("Python")?
  A. 5
  B. 6
  C. 7
  D. Error
Your answer: A
Wrong! The correct answer is B.

Question 5 of 6
A tuple can be changed after it is created.
  T = True, F = False
